# Notebook 03 — Wealth UI

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

How does the same backbone serve a structurally different application?

This notebook answers that question with working code. By the end, you will have
seen how the Wealth UI queries the same GraphQL schema as the Wholesale UI but
renders a different lens — and you will understand why this validates Thesis 2
of the ATLAS architecture.

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
# Skips uv sync (which installs the full agent stack and takes minutes).
import sys, subprocess

# Only install packages not already provided by the SageMaker base image
pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0']

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Thesis 2** | The ATLAS architectural thesis that two structurally different UIs can consume the same backbone (GraphQL schema, agent registry, MCP servers) by selecting different fragments and capability palettes based on persona. |
| **GraphQL fragment** | A named subset of fields from a GraphQL type. The Wholesale UI and Wealth UI query the same `Customer` type but select different fragments — the Wholesale UI selects referral-relevant fields; the Wealth UI selects coverage and theme fields. |
| **Capability palette** | The set of agent capabilities visible to a persona. The Consumer Banker sees referral-oriented capabilities; the Wealth Advisor sees coverage, themes, and conversational capabilities. Same registry, different views. |
| **Theme-summarizer** | A Phase 2 agent that produces natural-language summaries of market themes relevant to a client's portfolio. Accessible only to the Wealth Advisor persona. |
| **Conversational surface** | The Wealth UI component that asks the conversational-context-manager a question. It is honestly **single-turn**: each question is answered independently (`priorTurns` is always 0 because AgentCore Memory is not wired). Not present in the Wholesale UI. |
| **"New — routed to you" banner** | The flag the Wealth UI shows on a client the routing workflow just handed to this advisor. It is a *real* state — shown while `routedByWorkflow && !takenOnAt` — not a timer. **Take on client** writes a real `atlas:takenOnAt` and the banner clears. It is parallel to coverage: taking on does not change `isActive` or `coverageStartDate`. |
| **Lens** | A persona-specific view of the same underlying data. The Wholesale UI lens shows referral signals; the Wealth UI lens shows client coverage, behavioral signals, and themes. |

## Same schema, different lens

The Wholesale UI built in Phase 1 queries the ATLAS GraphQL schema to show a
Consumer Banker their referral candidates: customers with wealth signals, coverage
gaps, and household context. The Wealth UI built in Phase 2 queries the same
schema — the same AppSync endpoint, the same resolvers, the same underlying
Neptune graph — but renders a completely different application. Where the Wholesale
UI shows a referral pipeline, the Wealth UI shows a client coverage dashboard with
behavioral signals, market themes, and a conversational surface for follow-up
questions.

This is possible because the GraphQL schema is persona-neutral at the type level.
The `Customer` type has fields for both referral data (signals, household members)
and wealth data (AUM, themes, engagement metrics). Each UI selects only the fields
it needs through GraphQL fragments. The Wholesale UI's `CustomerReferralFragment`
selects `signals`, `household`, and `coverageGap`. The Wealth UI's
`CustomerCoverageFragment` selects `aum`, `themes`, `engagementScore`, and
`behavioralSignals`. Both fragments query the same type; they just ask for
different columns.

The capability palette works the same way. When the Wholesale UI calls
`capabilities(personaClaim: "atlas-consumer-banker")`, the registry returns
referral-oriented agents: nl-to-sparql-agent, wealth-signal-detector,
household-traverser, referral-rationale-drafter, referral-orchestrator. When the
Wealth UI calls `capabilities(personaClaim: "atlas-wealth-advisor")`, the registry
returns a different set: nl-to-sparql-agent, behavioral-signal-agent,
theme-summarizer, conversational-context-manager. Same registry endpoint, same
query shape, different results — because the registry filters by persona claim.

This is Thesis 2: the backbone (schema + registry + MCP servers) is shared
infrastructure. The UIs are thin clients that select their view through fragments
and persona claims. Adding a third UI — say, a compliance dashboard for BSA
Analysts — would not require changing the schema, the registry, or the MCP
servers. It would require only a new set of fragments and a new persona claim.
The architecture scales by addition, not modification.

In [ ]:
import sys
import os
import json

# Workshop 1's shared helpers.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

from pathlib import Path

SPEC_DIR = "../../spec/04-aws-agent-registry"
SCHEMA_PATH = "../../spec/05-appsync-graphql/schema.graphql"

def load_descriptors(subdir):
    path = os.path.join(SPEC_DIR, subdir)
    descriptors = []
    if not os.path.isdir(path):
        print(f"WARNING: {path} not found.")
        return descriptors
    for f in sorted(os.listdir(path)):
        if f.endswith(".json"):
            with open(os.path.join(path, f)) as fh:
                descriptors.append(json.load(fh))
    return descriptors

agent_descriptors = load_descriptors("agents")

print(f"Agents loaded: {len(agent_descriptors)}")
print("Setup complete.")

In [ ]:
# Build cell 1 — Same GraphQL query, different persona, different results.
#
# Simulate the capabilities query for both personas to show how the
# same endpoint returns different capability palettes.

def capabilities_query(persona_claim, descriptors):
    """Simulate the capabilities(personaClaim) GraphQL resolver."""
    return [
        {
            "agent_name": d["agent_name"],
            "description": d.get("description", ""),
            "posture": d.get("posture", "unknown"),
        }
        for d in descriptors
        if persona_claim in d.get("registry_metadata", {}).get("discoverable_by", [])
    ]

# Consumer Banker (Wholesale UI)
banker_caps = capabilities_query("atlas-consumer-banker", agent_descriptors)
print("Wholesale UI — Consumer Banker capabilities:")
print("-" * 50)
for cap in banker_caps:
    print(f"  {cap['agent_name']:<35} ({cap['posture']})")

print()

# Wealth Advisor (Wealth UI)
advisor_caps = capabilities_query("atlas-wealth-advisor", agent_descriptors)
print("Wealth UI — Wealth Advisor capabilities:")
print("-" * 50)
for cap in advisor_caps:
    print(f"  {cap['agent_name']:<35} ({cap['posture']})")

print()
print(f"Consumer Banker sees {len(banker_caps)} capabilities.")
print(f"Wealth Advisor sees {len(advisor_caps)} capabilities.")

In [ ]:
# Build cell 2 — Demonstrate theme-summarizer (Wealth UI only).
#
# The theme-summarizer produces natural-language summaries of market
# themes relevant to a client's portfolio. Only the Wealth Advisor
# persona can discover and invoke it.

def theme_summarizer(client_uri, themes_data):
    """Simulate theme-summarizer agent output.
    
    In production, this calls Bedrock to generate a summary
    from structured theme data in the graph.
    """
    return {
        "client_uri": client_uri,
        "themes": [
            {
                "theme_name": t["name"],
                "relevance_score": t["score"],
                "summary": t["summary"],
            }
            for t in themes_data
        ],
        "is_probabilistic": True,
        "requires_human_review": False,
    }

sample_themes = [
    {"name": "ESG Transition", "score": 0.82, "summary": "Portfolio has 40% exposure to energy transition assets."},
    {"name": "Rate Sensitivity", "score": 0.71, "summary": "Fixed income allocation sensitive to rate changes."},
    {"name": "Tech Concentration", "score": 0.65, "summary": "Equity holdings concentrated in technology sector."},
]

result = theme_summarizer("atlas:client-rachel-kim", sample_themes)
print("Theme summarizer output (Wealth UI only):")
print(json.dumps(result, indent=2))

In [ ]:
# Build cell 3 — Show different GraphQL fragments for each UI.
#
# Same Customer type, different field selections.

WHOLESALE_FRAGMENT = """
fragment CustomerReferralFragment on Customer {
  uri
  customerId
  fullName
  signals { signalType, detectedDate, evidence }
  household { members { fullName, relationship } }
  coverageGap
}
"""

WEALTH_FRAGMENT = """
fragment CustomerCoverageFragment on Customer {
  uri
  customerId
  fullName
  aum
  themes { themeName, relevanceScore, summary }
  engagementScore
  behavioralSignals { signalType, decayRatio, fired }
  advisor { fullName, teamId }
}
"""

print("Wholesale UI fragment (Consumer Banker):")
print(WHOLESALE_FRAGMENT)
print("Wealth UI fragment (Wealth Advisor):")
print(WEALTH_FRAGMENT)
print("Same Customer type. Different fields. Different application.")

## The demo loop — Marcus's half (the "New — routed to you" banner → take-on → clears)

The Wealth UI is also where the second half of the workshop demo loop plays out. After
**Dana Brooks** routes a referral from the Wholesale UI (her half is in
[`../phase-1-referral/06_wholesale_ui.ipynb`](../phase-1-referral/06_wholesale_ui.ipynb)),
**Marcus Webb (the Wealth Advisor)** signs in here and sees the handoff land:

1. **The banner appears.** The routed client — the customer **Rachel Kim** — shows in
   Marcus's book flagged **"New — routed to you."** This is a *real* state, not a timer or a
   client-side dismiss: the UI shows it precisely while the relationship was created by the
   routing workflow (`routedByWorkflow` is true) **and** Marcus has not yet accepted it
   (`takenOnAt` is null). It is the advisor's inbox of governed handoffs.
2. **Take on client → the banner clears.** Marcus opens the client's **Client 360** and clicks
   **Take on client**. That writes a *real* `atlas:takenOnAt` timestamp (the `takeOnClient`
   mutation) — a genuine accept transition — and the banner clears because `!takenOnAt` is no
   longer true.
3. **It is parallel to coverage (the no-harm design).** Taking on a client writes *only*
   `takenOnAt`. It does **not** touch `isActive` or `coverageStartDate` — coverage was already
   active at routing. The banner is the "not yet personally acknowledged" signal layered on
   top of unchanged coverage, so the route→cover loop is untouched.
4. **Converse is single-turn.** From the Client 360 Marcus can ask the conversational surface
   a question (`converse` → conversational-context-manager). It is honestly single-turn —
   each question is answered independently (`priorTurns` is always 0; AgentCore Memory is not
   wired), surfaced in the data, not hidden.

The workshop **Reset** (on Dana's dashboard, `resetDemoRoutings`) deletes the demo-created
routed relationships and routing decisions — which also clears this banner, since it is
driven by those same relationships — returning the graph to its seed state for the next run.

The take-on/banner mechanics on the schema side (`routedByWorkflow`, `takenOnAt`,
`takeOnClient`) and the **full cross-persona walk** are in
[`05_end_to_end.ipynb`](./05_end_to_end.ipynb), the canonical demo script
[`../../DEMO.md`](../../DEMO.md), and the presenter runbook
[`07_demo_runbook.ipynb`](./07_demo_runbook.ipynb). Switching personas in the live demo is
just **Sign out** (top-right) → sign in as the other user.

## Verification

Two properties must hold: the Wealth Advisor sees different capabilities than the
Consumer Banker (proving the registry filters by persona), and themes are accessible
to the Wealth Advisor (proving the theme-summarizer is discoverable). If either
fails, the two-UI thesis does not hold.

In [ ]:
# Verification cell 1 — Wealth Advisor sees different capabilities than Consumer Banker.

print("Verifying capability palette differentiation...")
print()

banker_names = {c["agent_name"] for c in banker_caps}
advisor_names = {c["agent_name"] for c in advisor_caps}

print(f"Consumer Banker capabilities: {sorted(banker_names)}")
print(f"Wealth Advisor capabilities:  {sorted(advisor_names)}")
print()

banker_only = banker_names - advisor_names
advisor_only = advisor_names - banker_names
shared = banker_names & advisor_names

print(f"Consumer Banker only: {sorted(banker_only)}")
print(f"Wealth Advisor only:  {sorted(advisor_only)}")
print(f"Shared:               {sorted(shared)}")
print()

if banker_names == advisor_names:
    print("VERIFICATION FAILED: Both personas see identical capabilities.")
    print("The registry must filter by persona claim. Check that agent descriptors")
    print("have different discoverable_by lists for different personas.")

assert banker_names != advisor_names, (
    "Consumer Banker and Wealth Advisor must see different capability palettes. "
    "Check registry_metadata.discoverable_by in agent descriptors."
)

print("[PASS] Wealth Advisor sees different capabilities than Consumer Banker.")

In [ ]:
# Verification cell 2 — Themes are accessible to Wealth Advisor.

print("Verifying theme-summarizer accessibility...")
print()

theme_agent = next(
    (d for d in agent_descriptors if d["agent_name"] == "theme-summarizer"),
    None
)

if theme_agent is None:
    print("VERIFICATION FAILED: theme-summarizer agent not found.")
    assert False, "theme-summarizer descriptor missing from registry."

discoverable_by = theme_agent.get("registry_metadata", {}).get("discoverable_by", [])
print(f"theme-summarizer discoverable_by: {discoverable_by}")
print()

advisor_can_see = "atlas-wealth-advisor" in discoverable_by
banker_cannot_see = "atlas-consumer-banker" not in discoverable_by

print(f"Wealth Advisor can discover: {advisor_can_see}")
print(f"Consumer Banker cannot:      {banker_cannot_see}")
print()

if not advisor_can_see:
    print("VERIFICATION FAILED: theme-summarizer not discoverable by Wealth Advisor.")
    print("Add 'atlas-wealth-advisor' to registry_metadata.discoverable_by.")

assert advisor_can_see, (
    "theme-summarizer must be discoverable by atlas-wealth-advisor. "
    "Themes are a core Wealth UI capability."
)

print("[PASS] Themes are accessible to the Wealth Advisor persona.")
print("The Wealth UI can render market themes for client portfolios.")

## What just changed

You have seen Thesis 2 in action: the same GraphQL schema and agent registry serve
two structurally different UIs. The Wholesale UI renders a referral pipeline for
Consumer Bankers; the Wealth UI renders a client coverage dashboard with themes and
conversational capabilities for Wealth Advisors. Same backbone, different fragments,
different capability palettes.

The next notebook explains the authentication change that makes this possible at
the request level: switching from IAM-based auth (service-to-service) to JWT-based
auth (user-to-service), where the persona claim travels as a token claim rather
than an IAM role assumption.